# Responses API: Group Chat Quarterly Planning

This notebook mirrors the `group-chat-quarterly-planning` Python scenario and breaks the workflow into runnable learning sections. It uses the Responses API shape: the client sends a normal `input`, while the server selects the scenario at startup.

## 1. Notebook Setup

Add this project's `src` directory to the Python path so the local package imports work from Jupyter.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start):
    current = Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "release_room").exists():
            return candidate
        nested = candidate / "responses-api-release-room"
        if (nested / "src" / "release_room").exists():
            return nested
    raise RuntimeError("Could not find responses-api-release-room project root.")

PROJECT_ROOT = find_project_root(Path.cwd())
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

PROJECT_ROOT

## 2. Load The Scenario And Helpers

The scenario module contains the enterprise use case, orchestration pattern, sample input, and agent roster.

In [ ]:
from release_room.scenarios.group_chat_quarterly_planning import SCENARIO
from release_room.scenarios import SCENARIOS_BY_ID, get_scenario
from release_room.notebook_helpers import (
    agent_roster,
    default_ollama_kwargs,
    pattern_anatomy,
    responses_api_reference,
    scenario_summary,
    workflow_result_to_text,
)

scenario = SCENARIO
assert SCENARIOS_BY_ID["group-chat-quarterly-planning"] is scenario
assert get_scenario("group-chat-quarterly-planning") is scenario

scenario_summary(scenario)

## 3. Pattern Anatomy

This explains what the orchestration pattern controls and when it fits enterprise application work.

In [ ]:
pattern_anatomy(scenario)

## 4. Flow Diagram

This runtime Mermaid diagram shows the API boundary, orchestration pattern, decisions, actions, and how the scenario agents interact. The helper returns the Mermaid source and displays a Mermaid image in Jupyter.


In [ ]:
from release_room.diagram_helpers import display_scenario_flow, scenario_flow_diagram

flow_diagram = display_scenario_flow(scenario)
flow_diagram.mermaid


## 5. Agent Roster

Each scenario uses multiple enterprise-role agents. Read the instructions to see what each specialist contributes.

In [ ]:
agent_roster(scenario)

## 6. Configure Ollama

The defaults keep local multi-agent runs predictable. Make sure Ollama is running and `qwen3:14b` is pulled.

In [ ]:
from release_room.agents import build_ollama_config

ollama_kwargs = default_ollama_kwargs()
config = build_ollama_config(**ollama_kwargs)
config

## 7. Build The Workflow

Responses API scenarios are selected at server startup. In-process notebooks pass the same scenario id directly to the workflow builder.

In [ ]:
from release_room.workflows import build_release_workflow

workflow = build_release_workflow(
    "group-chat-quarterly-planning",
    model=config.model,
    ollama_host=config.host,
    temperature=config.temperature,
    num_ctx=config.num_ctx,
    max_tokens=config.max_tokens,
    keep_alive=config.keep_alive,
    think=config.think,
)
workflow

## 8. Live In-Process Run

This runs the same orchestration without starting the HTTP server. It is the fastest way to study the agent behavior.

In [ ]:
result = await workflow.run(scenario.sample_input)
print(workflow_result_to_text(result)[:5000])

## 9. Responses API Boundary

This shows how the same scenario is exposed through the OpenAI-compatible `/responses` endpoint.

In [ ]:
responses_api_reference(scenario)

## 10. Experiments

Try changing `scenario.sample_input`, reducing `max_tokens`, or starting the server command from the prior cell and sending the sample JSON from the `samples/` folder.